**TẠO DANH SÁCH LƯỢT NGHE CỦA TỪNG NGHỆ SĨ**

In [30]:
import pandas as pd
# Không cần import MinMaxScaler hay numpy nữa vì hàm rank() đã có sẵn trong pandas

file_path = '/kaggle/input/datasets/js042710/mudataset/mudataset.csv'
df = pd.read_csv(file_path)

# ==========================================
# 1. TÁCH TÊN NGHỆ SĨ COLLAB (Giữ nguyên - Rất chuẩn)
# ==========================================
df['artist_name'] = df['artist_name'].astype(str)
df['artist_name'] = df['artist_name'].str.replace(r'(?i)\s*(?:&|feat\.?|ft\.?|/|;|\bx\b)\s*', ',', regex=True)
df['artist_name'] = df['artist_name'].str.split(',')
df = df.explode('artist_name')
df['artist_name'] = df['artist_name'].str.strip()
df = df[df['artist_name'] != '']
# ==========================================

# 2. Dùng groupby để nhóm các nghệ sĩ lại và đếm số lượt stream
artist_stats = df.groupby('artist_name').size().reset_index(name='Streams')

# ==========================================
# 3. CÁCH TÍNH MỚI: XẾP HẠNG PHẦN TRĂM (PERCENTILE RANK)
# ==========================================
# pct=True sẽ tự động biến thứ hạng thành thang điểm từ 0.0 -> 1.0
# Nó trả lời câu hỏi: "Nghệ sĩ này đánh bại bao nhiêu % nghệ sĩ khác trên thị trường?"
artist_stats['Popularity_Score'] = artist_stats['Streams'].rank(pct=True)
# ==========================================

# 4. Sắp xếp các bảng theo Lượt nghe từ cao xuống thấp
artist_stats = artist_stats.sort_values(by='Streams', ascending=False).reset_index(drop=True)

# 5. Thêm cột số thứ tự
artist_stats['Rank'] = artist_stats.index + 1

# Chọn lại các cột cần thiết (Đổi tên cột cuối thành Popularity_Score)
artist_stats = artist_stats[['Rank', 'artist_name', 'Streams', 'Popularity_Score']]

# 6. Xuất kết quả ra file xlsx mới 
output_file_excel = "bang_luot_nghe_nghe_si.xlsx"
artist_stats.to_excel(output_file_excel, index=False, engine='openpyxl')

# In ra 10 dòng đầu để kiểm tra
print(artist_stats.head(10))

   Rank            artist_name  Streams  Popularity_Score
0     1           Modest Mouse     8924          1.000000
1     2              Radiohead     8594          0.999993
2     3     The Mountain Goats     8384          0.999986
3     4            Duran Duran     6293          0.999979
4     5            The Beatles     6248          0.999972
5     6             Aphex Twin     5917          0.999965
6     7       Boards of Canada     5631          0.999958
7     8        Nine Inch Nails     5598          0.999952
8     9             Ben Harper     4961          0.999945
9    10  The New Pornographers     4947          0.999938


In [29]:
# !rm -rf /kaggle/working/*
# !pip install openpyxl

In [32]:
import pandas as pd
import re

# ---------------------------------------------------------
# 1. ĐỌC DỮ LIỆU ĐẦU VÀO
# ---------------------------------------------------------
file_log_path = '/kaggle/input/datasets/js042710/mudataset/mudataset.csv'
df_log = pd.read_csv(file_log_path)

path_lis = '/kaggle/input/datasets/js042710/bangluotnghenghesi/bang_luot_nghe_nghe_si.xlsx'
df_dict = pd.read_excel(path_lis)

# ---------------------------------------------------------
# 2. ĐỒNG BỘ: TÁCH NGHỆ SĨ COLLAB TRONG FILE LOG GỐC
# ---------------------------------------------------------
print("Đang tách tên nghệ sĩ collab trong file log...")
df_log['artist_name'] = df_log['artist_name'].astype(str)
df_log['artist_name'] = df_log['artist_name'].str.replace(r'(?i)\s*(?:&|feat\.?|ft\.?|/|;|\bx\b)\s*', ',', regex=True)
df_log['artist_name'] = df_log['artist_name'].str.split(',')
df_log = df_log.explode('artist_name')
df_log['artist_name'] = df_log['artist_name'].str.strip()
df_log = df_log[df_log['artist_name'] != '']

# ---------------------------------------------------------
# 3. GHÉP BẢNG (VLOOKUP) ĐỂ TÌM ĐIỂM SỐ
# ---------------------------------------------------------
print("Đang ghép điểm số từ từ điển vào lịch sử nghe...")
# CẬP NHẬT: Thay 'MinMaxScaler Score' thành 'Popularity_Score'
df_merged = df_log.merge(df_dict[['artist_name', 'Popularity_Score']], on='artist_name', how='inner')

# ---------------------------------------------------------
# 4. TÍNH TOÁN ĐIỂM GU ÂM NHẠC CHO TỪNG USER
# ---------------------------------------------------------
print("Đang tính toán Điểm Gu Âm Nhạc (Taste Score)...")
# CẬP NHẬT: Tính trung bình theo cột 'Popularity_Score'
user_taste = df_merged.groupby('user_id')['Popularity_Score'].mean().reset_index()

# Đổi tên cột cho chuẩn chỉnh
user_taste.columns = ['user_id', 'Taste_Score']

# Sắp xếp từ cao xuống thấp để xem ai "bắt trend" nhất
user_taste = user_taste.sort_values(by='Taste_Score', ascending=False).reset_index(drop=True)

# ---------------------------------------------------------
# 5. XUẤT FILE KẾT QUẢ GIAI ĐOẠN 3
# ---------------------------------------------------------
output_file = "diem_gu_am_nhac_user.csv"
user_taste.to_csv(output_file, index=False, encoding='utf-8-sig')

print("\nHOÀN TẤT GIAI ĐOẠN 3! XEM THỬ 10 USER ĐẦU TIÊN:")
print(user_taste.head(10))

Đang tách tên nghệ sĩ collab trong file log...
Đang ghép điểm số từ từ điển vào lịch sử nghe...
Đang tính toán Điểm Gu Âm Nhạc (Taste Score)...

HOÀN TẤT GIAI ĐOẠN 3! XEM THỬ 10 USER ĐẦU TIÊN:
   user_id  Taste_Score
0    50174     0.999993
1    88675     0.999993
2    74292     0.999993
3    28979     0.999993
4    80363     0.999993
5    80928     0.999993
6    18802     0.999986
7    77379     0.999979
8    22319     0.999979
9    51000     0.999979


In [34]:
import pandas as pd
from sklearn.cluster import KMeans
import numpy as np

# 1. Đọc dữ liệu từ Giai đoạn 3 (Bảng chứa user_id và Taste_Score)
file_path = '/kaggle/input/datasets/js042710/diemguamnhac/diem_gu_am_nhac_user.csv' # Thay đổi tên nếu file của bạn tên khác
user_taste = pd.read_csv(file_path)

# 2. Chuẩn bị dữ liệu cho K-Means (Chuyển thành mảng 2D)
X = user_taste['Taste_Score'].values.reshape(-1, 1)

# 3. Khởi tạo và chạy thuật toán K-Means với 3 cụm (k=3)
print("Đang cho AI chạy phân nhóm K-Means...")
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
user_taste['Cluster_ID'] = kmeans.fit_predict(X)

# ==========================================
# 4. ĐỊNH DANH CÁC CỤM (GẮN NHÃN)
# ==========================================
# Lấy giá trị của 3 tâm cụm (Centroids)
centers = kmeans.cluster_centers_.flatten()

# Sắp xếp các tâm cụm từ thấp đến cao để biết ai là ngách, ai là trend
# index_sorted[0]: là ID của cụm có tâm thấp nhất -> Niche
# index_sorted[1]: là ID của cụm có tâm ở giữa -> Average
# index_sorted[2]: là ID của cụm có tâm cao nhất -> Mainstream
index_sorted = np.argsort(centers)

# Tạo từ điển ánh xạ (Mapping)
label_mapping = {
    index_sorted[0]: 'Niche',
    index_sorted[1]: 'Average',
    index_sorted[2]: 'Mainstream'
}

# Gắn nhãn chữ vào bảng dữ liệu
user_taste['Taste_Label'] = user_taste['Cluster_ID'].map(label_mapping)

# Dọn dẹp: Bỏ cột Cluster_ID bằng số đi cho gọn
user_taste = user_taste.drop(columns=['Cluster_ID'])

# ==========================================
# 5. THỐNG KÊ VÀ LƯU KẾT QUẢ
# ==========================================
print("\n📊 THỐNG KÊ SỐ LƯỢNG NGƯỜI DÙNG THEO GU ÂM NHẠC:")
print(user_taste['Taste_Label'].value_counts())

# Lưu file MASTER TABLE cho Module 2
output_file = "Final_Module2_User_Taste.csv"
user_taste.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\n🎉 HOÀN TẤT MODULE 2! Đã lưu file dữ liệu cuối cùng tại: {output_file}")
print("\nXEM THỬ 10 DÒNG KẾT QUẢ:")
print(user_taste.head(10))

Đang cho AI chạy phân nhóm K-Means...

📊 THỐNG KÊ SỐ LƯỢNG NGƯỜI DÙNG THEO GU ÂM NHẠC:
Taste_Label
Mainstream    8208
Average       4154
Niche         1005
Name: count, dtype: int64

🎉 HOÀN TẤT MODULE 2! Đã lưu file dữ liệu cuối cùng tại: Final_Module2_User_Taste.csv

XEM THỬ 10 DÒNG KẾT QUẢ:
   user_id  Taste_Score Taste_Label
0    50174     0.999993  Mainstream
1    88675     0.999993  Mainstream
2    74292     0.999993  Mainstream
3    28979     0.999993  Mainstream
4    80363     0.999993  Mainstream
5    80928     0.999993  Mainstream
6    18802     0.999986  Mainstream
7    77379     0.999979  Mainstream
8    22319     0.999979  Mainstream
9    51000     0.999979  Mainstream
